# 1. Time-frequency processing and tremor characteristics

What this notebook does:

1. computes the frequency characteristics of every patient's tremor
2. compares them across N / PD / ET
3. classifies patients from **mean and max frequency alone**, then adds one
   characteristic at a time so each contribution is visible

Run top to bottom. Needs `Data/`, `NewData/`, `pads_stretchhold/`.


In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, matplotlib.pyplot as plt
from tfbench.characteristics import (patient_table, describe, classify,
                                     FEATURES, CLASS_NAMES)
from tremor.quaternion_data import load_quaternion_recordings
from pdetn.load_2025 import load_2025_all
from pdetn.crossdataset import load_pads_extracted

## Load the three cohorts

In [ ]:
cohorts = {
    '2015 OUT':         (load_quaternion_recordings('Data', action='OUT',
                                                    mode='angular_velocity'), slice(3, 6)),
    'NewData OUT':      (load_2025_all(conditions=('OUT',)), slice(3, 6)),
    'PADS StretchHold': (load_pads_extracted('pads_stretchhold'), slice(0, 3)),
}
tables = {n: patient_table(r, ch=c) for n, (r, c) in cohorts.items()}
for n, (X, y, _) in tables.items():
    print(f"{n:>18}  n={len(y):>4}  " +
          "  ".join(f"{c}={int((y==i).sum())}" for i, c in enumerate(CLASS_NAMES)))

## Tremor characteristics per class

`max_freq` dominant frequency &nbsp;|&nbsp; `mean_freq` spectral centroid &nbsp;|&nbsp;
`bandwidth` spread about the centroid &nbsp;|&nbsp; `inband_frac` 3-15 Hz share of
total power &nbsp;|&nbsp; `harm_ratio` power at 2f &nbsp;|&nbsp; `peak_sharp` peak
over mean in-band power.

In [ ]:
for n, (X, y, _) in tables.items():
    describe(X, y, tag=n)

## Distributions — where each class sits in frequency

In [ ]:
X, y, _ = tables['PADS StretchHold']          # largest cohort
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for j, name in enumerate(FEATURES):
    a = ax[j // 3][j % 3]
    for i, c in enumerate(CLASS_NAMES):
        v = X[y == i, j]
        v = v[np.isfinite(v)]
        if len(v):
            a.hist(v, bins=18, alpha=0.5, label=f'{c} (n={len(v)})', density=True)
    a.set_title(name); a.legend(fontsize=7)
fig.suptitle('PADS StretchHold — tremor characteristics by class')
fig.tight_layout(); plt.show()

## Classify from frequency alone

Features are added cumulatively, starting from `max_freq`, so the marginal value
of each one is visible.

In [ ]:
for n, (X, y, _) in tables.items():
    classify(X, y, tag=n, axis='N_vs_Tremor')
    classify(X, y, tag=n, axis='PD_vs_ET')

## What this shows (measured, see `reports/tremor_characteristics.md`)

* **N-vs-Tremor reaches precision 0.91-0.92 from six interpretable frequency
  numbers** — 2015 AUC 0.890 / precision 0.910, PADS AUC 0.804 / precision 0.924.
* **`peak_sharp` is the strongest single characteristic**: ET tremor is far more
  sharply peaked than PD (PADS: ET 12.2, PD 5.8, N 4.1). ET is close to a pure
  tone, PD is broader.
* **`mean_freq` adds most to PD-vs-ET on PADS**: AUC 0.649 -> 0.786 when added
  to `max_freq`, close to the full 10-descriptor set (0.807).
* **PD-vs-ET on 2015 is below chance from frequency alone** (AUC ~0.31). The
  frequency route works on PADS and fails on 2015 — see
  `reports/temporal_stability.md`, where instantaneous-frequency stability is
  the feature family that does work there.
